# Evaluación Comparativa Final — HICP Eurozona

**Pregunta central**: ¿Mejoran las señales MCP (BCE/GDELT) e institucionales (DFR, ESI, Brent, TTF, EPU) la predicción del HICP europeo en foundation models de series temporales?

Backtesting rolling-origin 2021-2024 · 18 modelos · Horizontes h=1,3,6,12 meses

**Secciones:**
1. ¿Qué modelo predice mejor en cada horizonte?
2. ¿Las señales C1 mejoran o empeoran frente a la condición univariante (C0)?
3. ¿Hay diferencias por periodo (pre-shock / shock / normalización)?
4. ¿Son las diferencias estadísticamente significativas (test Diebold-Mariano)?
5. **Figura principal** para la memoria del TFG

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
RESULTS = ROOT / '08_results'
HORIZONS = [1, 3, 6, 12]

# ── Cargar métricas desde ficheros individuales ────────────────────
baseline_raw = json.load(open(RESULTS / 'rolling_metrics_europe.json'))
deep_raw     = json.load(open(RESULTS / 'deep_rolling_metrics_europe.json'))

FOUNDATION_MODELS = [
    'chronos2_C0_europe', 'chronos2_C1_inst_europe',
    'chronos2_C1_mcp_europe', 'chronos2_C1_full_europe',
    'timesfm_C0_europe',  'timesfm_C1_inst_europe',
    'timesfm_C1_mcp_europe', 'timesfm_C1_full_europe',
    'timegpt_C0_europe',  'timegpt_C1_inst_europe',
    'timegpt_C1_mcp_europe', 'timegpt_C1_full_europe',
]

metrics = {}
# Baseline (sin sufijo _europe en el JSON)
for k, v in baseline_raw.items():
    metrics[f'{k}_europe'] = v
# Deep (sin sufijo _europe en el JSON)
for k, v in deep_raw.items():
    metrics[f'{k}_europe'] = v
# Foundation
for name in FOUNDATION_MODELS:
    p = RESULTS / f'{name}_metrics.json'
    if p.exists():
        d = json.load(open(p))
        metrics[name] = d[name]

# DM tests
dm_raw = json.load(open(RESULTS / 'diebold_mariano_results_europe.json'))

print(f'Modelos cargados: {len(metrics)}')
print(f'Pares DM: {len(dm_raw)}')

In [ ]:
# ── Paleta y orden ─────────────────────────────────────────────────
FAMILY_COLORS = {
    'naive_europe':   '#b0b0b0', 'sarima_europe': '#606060', 'sarimax_europe': '#404040',
    'auto_arima_europe': '#8B4513',
    'lstm_europe':    '#4e9af1', 'nbeats_europe': '#1a6ab0', 'nhits_europe': '#0d3d6e',
    'chronos2_C0_europe':       '#52a852',
    'chronos2_C1_inst_europe':  '#c5b0d5',
    'chronos2_C1_mcp_europe':   '#e6841a',
    'chronos2_C1_full_europe':  '#a0522d',
    'timesfm_C0_europe':        '#2ca02c',
    'timesfm_C1_inst_europe':   '#9467bd',
    'timesfm_C1_mcp_europe':    '#ff7f0e',
    'timesfm_C1_full_europe':   '#17becf',
    'timegpt_C0_europe':        '#8fbc8f',
    'timegpt_C1_inst_europe':   '#d0a0e0',
    'timegpt_C1_mcp_europe':    '#ffc070',
    'timegpt_C1_full_europe':   '#90e0ef',
}

MODEL_ORDER = [
    'naive_europe', 'sarima_europe', 'sarimax_europe', 'auto_arima_europe',
    'lstm_europe', 'nbeats_europe', 'nhits_europe',
    'chronos2_C0_europe', 'chronos2_C1_inst_europe',
    'chronos2_C1_mcp_europe', 'chronos2_C1_full_europe',
    'timesfm_C0_europe',  'timesfm_C1_inst_europe',
    'timesfm_C1_mcp_europe', 'timesfm_C1_full_europe',
    'timegpt_C0_europe',  'timegpt_C1_inst_europe',
    'timegpt_C1_mcp_europe', 'timegpt_C1_full_europe',
]
MODEL_ORDER = [m for m in MODEL_ORDER if m in metrics]

# ── DataFrame plano de métricas ────────────────────────────────────
rows = []
for model in MODEL_ORDER:
    for h in HORIZONS:
        m = metrics[model].get(f'h{h}', {})
        rows.append({'model': model, 'horizon': h,
                     'MAE': m.get('MAE'), 'RMSE': m.get('RMSE'),
                     'MASE': m.get('MASE'), 'n': m.get('n_evals')})
df = pd.DataFrame(rows)

# ── DataFrame plano de DM tests ────────────────────────────────────
dm_records = []
for entry in dm_raw:
    for h in HORIZONS:
        hd = entry.get(f'h{h}', {})
        dm_records.append({
            'model1': entry['model1'], 'model2': entry['model2'],
            'period': entry['period'], 'horizon': h,
            'dm_stat': hd.get('dm_stat'), 'p_value': hd.get('p_value'),
            'better': hd.get('better'), 'n': hd.get('n', 0),
        })
dm_df = pd.DataFrame(dm_records)

print(f'df: {df.shape} | dm_df: {dm_df.shape}')
# AutoARIMA en el ranking
aa = metrics.get('auto_arima_europe', {})
print(f'AutoARIMA Europe  h1={aa.get("h1",{}).get("MAE","?")}  h12={aa.get("h12",{}).get("MAE","?")}')
sarima = metrics.get('sarima_europe', {})
print(f'SARIMA Europe     h1={sarima.get("h1",{}).get("MAE","?")}  h12={sarima.get("h12",{}).get("MAE","?")}')

---
## 1 · ¿Qué modelo predice mejor el HICP en cada horizonte?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=False)

# Ordenar por MAE h=1
sorted_mods = (df[(df['horizon']==1) & (df['model']!='naive_europe')]
               .set_index('model')['MAE']
               .reindex([m for m in MODEL_ORDER if m != 'naive_europe'])
               .dropna().sort_values().index.tolist())

for ax, h in zip(axes, HORIZONS):
    sub = (df[(df['horizon']==h) & (df['model'].isin(sorted_mods))]
           .set_index('model').reindex(sorted_mods))
    colors = [FAMILY_COLORS.get(m, '#999') for m in sorted_mods]
    ax.barh(range(len(sorted_mods)), sub['MAE'], color=colors, edgecolor='white', lw=0.4)
    best_idx = sub['MAE'].dropna().idxmin()
    ax.barh(sorted_mods.index(best_idx), sub.loc[best_idx,'MAE'],
            color=FAMILY_COLORS.get(best_idx,'#999'), edgecolor='gold', lw=2.5)
    ax.set_yticks(range(len(sorted_mods)))
    ax.set_yticklabels([m.replace('_europe','') for m in sorted_mods], fontsize=7.5)
    ax.set_xlabel('MAE (puntos HICP)', fontsize=9)
    ax.set_title(f'h={h}', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

legend_items = [
    mpatches.Patch(color='#606060', label='Estadístico (SARIMA)'),
    mpatches.Patch(color='#1a6ab0', label='Deep Learning'),
    mpatches.Patch(color='#2ca02c', label='Foundation C0 (univariante)'),
    mpatches.Patch(color='#9467bd', label='Foundation C1_inst (DFR+ESI+Brent…)'),
    mpatches.Patch(color='#ff7f0e', label='Foundation C1_mcp (BCE+GDELT)'),
    mpatches.Patch(color='#17becf', label='Foundation C1_full (inst+mcp)'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=3, fontsize=8,
           frameon=True, bbox_to_anchor=(0.5, -0.06))
fig.suptitle('MAE por modelo y horizonte — HICP Eurozona · Rolling-origin 2021-2024\n'
             '(borde dorado = mejor de ese horizonte | naive excluido del ranking)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / 'fig1_mae_all_models_europe.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop-5 por horizonte:')
for h in HORIZONS:
    top = df[(df['horizon']==h) & (df['model']!='naive_europe')].sort_values('MAE').head(5)
    print(f'h={h}: ' + ' | '.join(
        f"{r['model'].replace('_europe',''):30s} {r['MAE']:.4f}" for _, r in top.iterrows()))

---
## 2 · ¿Las señales C1 mejoran la predicción? (Δ MAE relativo vs C0)

In [ ]:
C1_PAIRS = [
    ('Chronos-2 C1_inst', 'chronos2_C0_europe', 'chronos2_C1_inst_europe'),
    ('Chronos-2 C1_mcp',  'chronos2_C0_europe', 'chronos2_C1_mcp_europe'),
    ('Chronos-2 C1_full', 'chronos2_C0_europe', 'chronos2_C1_full_europe'),
    ('TimesFM  C1_inst ★','timesfm_C0_europe',  'timesfm_C1_inst_europe'),
    ('TimesFM  C1_mcp',   'timesfm_C0_europe',  'timesfm_C1_mcp_europe'),
    ('TimesFM  C1_full ★','timesfm_C0_europe',  'timesfm_C1_full_europe'),
    ('TimeGPT  C1_inst',  'timegpt_C0_europe',  'timegpt_C1_inst_europe'),
    ('TimeGPT  C1_mcp',   'timegpt_C0_europe',  'timegpt_C1_mcp_europe'),
    ('TimeGPT  C1_full',  'timegpt_C0_europe',  'timegpt_C1_full_europe'),
]

hm = np.full((len(C1_PAIRS), 4), np.nan)
for i, (lbl, c0n, c1n) in enumerate(C1_PAIRS):
    for j, h in enumerate(HORIZONS):
        m0 = metrics.get(c0n, {}).get(f'h{h}', {}).get('MAE')
        m1 = metrics.get(c1n, {}).get(f'h{h}', {}).get('MAE')
        if m0 and m1:
            hm[i, j] = (m1 - m0) / m0 * 100

fig, ax = plt.subplots(figsize=(8, 7))
norm = TwoSlopeNorm(vmin=-15, vcenter=0, vmax=100)
im = ax.imshow(hm, cmap='RdYlGn_r', norm=norm, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(C1_PAIRS)))
ax.set_yticklabels([r[0] for r in C1_PAIRS], fontsize=9)
ax.set_title('Δ MAE relativo: (C1 − C0) / C0 × 100%\n'
             'Verde = C1 mejor  |  Rojo = C0 mejor  |  ★ = condición que mejora o empata',
             fontsize=10, fontweight='bold')
for i in range(len(C1_PAIRS)):
    for j in range(4):
        v = hm[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 35 else 'black'
            ax.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                    fontsize=9, color=ct, fontweight='bold')
# Separadores de familia
for b in [3, 6]:
    ax.axhline(b - 0.5, color='white', linewidth=2.5)
plt.colorbar(im, ax=ax, shrink=0.65, label='Δ MAE (%)')
plt.tight_layout()
plt.savefig(RESULTS / 'fig2_delta_mae_heatmap_europe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Perfiles MAE: C0 vs mejor C1 por familia (+ AutoARIMA como referencia) ─
PAIRS_PROFILE = [
    ('Chronos-2', 'chronos2_C0_europe', 'chronos2_C1_full_europe',  '#52a852', '#a0522d'),
    ('TimesFM',   'timesfm_C0_europe',  'timesfm_C1_full_europe',   '#2ca02c', '#17becf'),
    ('TimeGPT',   'timegpt_C0_europe',  'timegpt_C1_mcp_europe',    '#8fbc8f', '#ffc070'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(HORIZONS))

for ax, (fname, c0, c1, col0, col1) in zip(axes, PAIRS_PROFILE):
    c0v = [metrics[c0][f'h{h}']['MAE'] for h in HORIZONS]
    c1v = [metrics[c1][f'h{h}']['MAE'] for h in HORIZONS]
    ax.plot(x, c0v, 'o-', color=col0, lw=2.5, ms=7, label=c0.replace('_europe',''), zorder=5)
    ax.plot(x, c1v, 's--', color=col1, lw=2.5, ms=7, label=c1.replace('_europe',''), zorder=5)
    ax.fill_between(x, c0v, c1v,
                    where=[v1 < v0 for v0, v1 in zip(c0v, c1v)],
                    alpha=0.18, color='green', label='C1 mejor')
    ax.fill_between(x, c0v, c1v,
                    where=[v1 >= v0 for v0, v1 in zip(c0v, c1v)],
                    alpha=0.12, color='red', label='C0 mejor')
    # Referencias
    sarima_v = [metrics['sarima_europe'][f'h{h}']['MAE'] for h in HORIZONS]
    ax.plot(x, sarima_v, ':', color='#888', lw=1.2, alpha=0.6, label='SARIMA')
    # AutoARIMA como referencia adicional
    if 'auto_arima_europe' in metrics:
        aa_v = [metrics['auto_arima_europe'][f'h{h}']['MAE'] for h in HORIZONS]
        ax.plot(x, aa_v, '--', color='#8B4513', lw=1.4, alpha=0.7, label='AutoARIMA')
    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}' for h in HORIZONS])
    ax.set_ylabel('MAE (puntos HICP)', fontsize=9)
    ax.set_title(fname, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7.5, loc='upper left')
    ax.grid(alpha=0.25)

fig.suptitle('C0 (univariante) vs mejor C1 por familia — Perfil MAE · HICP Eurozona\n'
             'AutoARIMA (marrón): selección dinámica de órdenes en cada origen',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig3_c0_vs_c1_profiles_europe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Error temporal mes a mes — ¿Cuándo falla cada modelo?

Error absoluto h=1 a lo largo de 2021-2024. Fondos sombreados:
**A** — pre-shock (2021-06), **B** — shock inflacionista (2022-07 a 2023-06), **C** — normalización (2023-07 a 2024-12).

In [ ]:
BASELINE_PREDS = RESULTS / 'rolling_predictions_europe.parquet'
DEEP_PREDS     = RESULTS / 'deep_rolling_predictions_europe.parquet'

_df_base = pd.read_parquet(BASELINE_PREDS)
_df_base['origin']  = pd.to_datetime(_df_base['origin'])
_df_base['fc_date'] = pd.to_datetime(_df_base['fc_date'])

_df_deep = pd.read_parquet(DEEP_PREDS)
_df_deep['origin']  = pd.to_datetime(_df_deep['origin'])
_df_deep['fc_date'] = pd.to_datetime(_df_deep['fc_date'])

def _load_foundation(name):
    d = pd.read_parquet(RESULTS / f'{name}_predictions.parquet')
    d['origin']  = pd.to_datetime(d['origin'])
    d['fc_date'] = pd.to_datetime(d['fc_date'])
    return d

def _get_h1_errors(name):
    base_name = name.replace('_europe', '')
    if base_name in ('lstm', 'nbeats', 'nhits'):
        sub = _df_deep[(_df_deep['model']==base_name) & (_df_deep['horizon']==1) & (_df_deep['step']==1)]
    elif base_name in ('sarima', 'arima', 'naive', 'sarimax'):
        sub = _df_base[(_df_base['model']==base_name) & (_df_base['horizon']==1) & (_df_base['step']==1)]
    else:
        d = _load_foundation(name)
        sub = d[(d['horizon']==1) & (d['step']==1)]
    return sub.set_index('fc_date')['abs_error'].sort_index()

SHADING = [
    ('A: Pre-shock',    '2021-01-01', '2022-06-30', '#d4e8d4', 0.45),
    ('B: Shock',        '2022-07-01', '2023-06-30', '#f8d7d7', 0.50),
    ('C: Normalización','2023-07-01', '2024-12-31', '#d7e8f8', 0.35),
]

TIMELINE_MODELS = [
    ('sarima_europe',           '#606060', 'SARIMA',                  '-',  1.5, 'o'),
    ('nbeats_europe',           '#1a6ab0', 'N-BEATS',                 '--', 1.8, 'D'),
    ('timesfm_C0_europe',       '#2ca02c', 'TimesFM C0',              '-',  2.0, 's'),
    ('chronos2_C0_europe',      '#52a852', 'Chronos-2 C0',            '-',  1.8, '^'),
    ('timesfm_C1_inst_europe',  '#9467bd', 'TimesFM C1_inst ★',       '-',  2.2, 'v'),
    ('timesfm_C1_full_europe',  '#17becf', 'TimesFM C1_full ★★',      '-',  2.0, 'P'),
]

fig, ax = plt.subplots(figsize=(14, 5))
for lbl, s, e, col, alpha in SHADING:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=alpha, zorder=0)

for name, col, label, ls, lw, mk in TIMELINE_MODELS:
    try:
        series = _get_h1_errors(name)
        if name == 'nbeats_europe':
            ax.scatter(series.index, series.values, color=col, marker=mk, s=60, zorder=6, label=label)
            ax.plot(series.index, series.values, ':', color=col, lw=1.0, alpha=0.45, zorder=5)
        else:
            ax.plot(series.index, series.values, ls=ls, color=col, lw=lw,
                    marker=mk, markersize=4, markevery=4, label=label, zorder=5)
    except Exception as ex:
        print(f'[!] {name}: {ex}')

ylim = ax.get_ylim()
for lbl, s, e, col, alpha in SHADING:
    ax.text(pd.Timestamp(s) + pd.Timedelta(days=22), ylim[1]*0.96,
            lbl, fontsize=8.5, color='#333', va='top', fontweight='bold')

ax.set_ylabel('Error absoluto h=1 (puntos HICP)', fontsize=10)
ax.set_xlabel('Fecha de predicción', fontsize=10)
ax.set_title('Error absoluto mes a mes — h=1 · HICP Eurozona 2021-2024', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
ax.grid(alpha=0.22, zorder=0)
ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-01-01'))
plt.tight_layout()
plt.savefig(RESULTS / 'fig3b_error_temporal_europe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Forecast fan — Chronos-2 C0 Europe (IC 80%) ────────────────────
df_ch = pd.read_parquet(RESULTS / 'chronos2_C0_europe_predictions.parquet')
df_ch['fc_date'] = pd.to_datetime(df_ch['fc_date'])
df_ch['origin']  = pd.to_datetime(df_ch['origin'])

h1_ch = df_ch[(df_ch['horizon']==1) & (df_ch['step']==1)].sort_values('fc_date').copy()
within_80 = ((h1_ch['y_true'] >= h1_ch['y_pred_p10']) &
             (h1_ch['y_true'] <= h1_ch['y_pred_p90'])).mean()
ic_width  = (h1_ch['y_pred_p90'] - h1_ch['y_pred_p10']).mean()
print(f'Cobertura IC 80%: {within_80:.1%} (nominal 80%)  |  Anchura media: {ic_width:.3f} pts')

fig, ax = plt.subplots(figsize=(14, 5))
for lbl, s, e, col, alpha in SHADING:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=alpha, zorder=0)

ax.fill_between(h1_ch['fc_date'], h1_ch['y_pred_p10'], h1_ch['y_pred_p90'],
                color='#52a852', alpha=0.22, label='IC 80% (p10-p90)', zorder=2)
ax.plot(h1_ch['fc_date'], h1_ch['y_pred_p10'], '--', color='#52a852', lw=0.9, alpha=0.6, zorder=3)
ax.plot(h1_ch['fc_date'], h1_ch['y_pred_p90'], '--', color='#52a852', lw=0.9, alpha=0.6, zorder=3)
ax.plot(h1_ch['fc_date'], h1_ch['y_pred'],    '-',  color='#2ca02c', lw=2.2,
        label='Chronos-2 C0 (mediana)', zorder=5)
ax.plot(h1_ch['fc_date'], h1_ch['y_true'],    'k-', lw=1.8, label='HICP real', zorder=6)
ax.scatter(h1_ch['fc_date'], h1_ch['y_true'], c='black', s=20, zorder=7)

outside = h1_ch[(h1_ch['y_true'] < h1_ch['y_pred_p10']) | (h1_ch['y_true'] > h1_ch['y_pred_p90'])]
if len(outside) > 0:
    ax.scatter(outside['fc_date'], outside['y_true'], c='#cc0000', s=80, zorder=8,
               marker='x', linewidths=2.2, label=f'Fuera IC 80% ({len(outside)} meses)')

ax.set_ylabel('Índice HICP Eurozona (base 2015=100)', fontsize=10)
ax.set_xlabel('Fecha de predicción', fontsize=10)
ax.set_title(
    f'Forecast Fan — Chronos-2 C0 · h=1 · IC 80% (p10-p90)\n'
    f'Cobertura empírica: {within_80:.1%} (nominal 80%) · Anchura media: {ic_width:.2f} puntos',
    fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', framealpha=0.88)
ax.grid(alpha=0.22, zorder=0)
ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-01-01'))
plt.tight_layout()
plt.savefig(RESULTS / 'fig3c_forecast_fan_europe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · Significancia estadística (DM global — C0 vs C1)

In [ ]:
C1_DM = [
    ('chronos2_C0_europe', 'chronos2_C1_inst_europe'),
    ('chronos2_C0_europe', 'chronos2_C1_mcp_europe'),
    ('chronos2_C0_europe', 'chronos2_C1_full_europe'),
    ('timesfm_C0_europe',  'timesfm_C1_inst_europe'),
    ('timesfm_C0_europe',  'timesfm_C1_mcp_europe'),
    ('timesfm_C0_europe',  'timesfm_C1_full_europe'),
    ('timegpt_C0_europe',  'timegpt_C1_inst_europe'),
    ('timegpt_C0_europe',  'timegpt_C1_mcp_europe'),
    ('timegpt_C0_europe',  'timegpt_C1_full_europe'),
]

dm_hm = np.full((len(C1_DM), 4), np.nan)
dm_pv = np.full((len(C1_DM), 4), np.nan)
for i, (m1, m2) in enumerate(C1_DM):
    sub = dm_df[(dm_df['model1']==m1) & (dm_df['model2']==m2) & (dm_df['period']=='global')]
    for j, h in enumerate(HORIZONS):
        hr = sub[sub['horizon']==h]
        if len(hr) and pd.notna(hr['dm_stat'].values[0]):
            dm_hm[i, j] = float(hr['dm_stat'].values[0])
            dm_pv[i, j] = float(hr['p_value'].values[0])

fig, ax = plt.subplots(figsize=(8, 7))
norm2 = TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)
im2 = ax.imshow(dm_hm, cmap='RdYlGn', norm=norm2, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=10)
ax.set_yticks(range(len(C1_DM)))
ylabels = [f"{m2.replace('_europe',''):30s}" for _, m2 in C1_DM]
ax.set_yticklabels(ylabels, fontsize=8.5)
ax.set_title('Estadístico DM: C0 vs C1 — período global\n'
             'Verde = C1 mejor  |  Rojo = C0 mejor  |  ** p<0.05  * p<0.10\n'
             'DM<0 ⟹ C0 mejor  |  DM>0 ⟹ C1 mejor',
             fontsize=10, fontweight='bold')
for i in range(len(C1_DM)):
    for j in range(4):
        if not np.isnan(dm_hm[i, j]):
            sig = '**' if dm_pv[i,j] < 0.05 else ('*' if dm_pv[i,j] < 0.10 else '')
            ct  = 'white' if abs(dm_hm[i,j]) > 2.8 else 'black'
            ax.text(j, i, f'{dm_hm[i,j]:.1f}{sig}', ha='center', va='center',
                    fontsize=9, color=ct)
for b in [3, 6]:
    ax.axhline(b - 0.5, color='white', lw=2.5)
plt.colorbar(im2, ax=ax, shrink=0.72, label='DM statistic')
plt.tight_layout()
plt.savefig(RESULTS / 'fig4_dm_global_heatmap_europe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── DM por subperiodo: TimesFM C0 vs C1_inst y C1_full ────────────
PERIODS = ['global', 'A_pre_shock', 'B_shock', 'C_normalizacion']
PERIOD_LABELS = {
    'global':          'Global\n2021-2024',
    'A_pre_shock':     'A: Pre-shock\n2021-06/22',
    'B_shock':         'B: Shock\n2022-07/23',
    'C_normalizacion': 'C: Normalización\n2023-07/24',
}

PAIRS_SUB = [
    ('timesfm_C0_europe', 'timesfm_C1_inst_europe', 'TimesFM C0 vs C1_inst'),
    ('timesfm_C0_europe', 'timesfm_C1_full_europe', 'TimesFM C0 vs C1_full'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (m1, m2, title) in zip(axes, PAIRS_SUB):
    sub = dm_df[(dm_df['model1']==m1) & (dm_df['model2']==m2)]
    xp = np.arange(len(PERIODS))
    w  = 0.18
    for i, h in enumerate(HORIZONS):
        dvals, pvals = [], []
        for p in PERIODS:
            row = sub[(sub['period']==p) & (sub['horizon']==h)]
            if len(row) and pd.notna(row['dm_stat'].values[0]):
                dvals.append(float(row['dm_stat'].values[0]))
                pvals.append(float(row['p_value'].values[0]))
            else:
                dvals.append(0.0); pvals.append(1.0)
        ax.bar(xp + (i-1.5)*w, dvals, w, label=f'h={h}',
               alpha=0.85, color=plt.cm.plasma(i/4))
        for xi, (dv, pv) in enumerate(zip(dvals, pvals)):
            if pv < 0.05:
                ax.text(xi+(i-1.5)*w, dv+(0.22 if dv>=0 else -0.55),
                        '**', ha='center', fontsize=9, fontweight='bold')
            elif pv < 0.10:
                ax.text(xi+(i-1.5)*w, dv+(0.22 if dv>=0 else -0.45),
                        '*', ha='center', fontsize=9)
    ax.axhline(0,     color='black',   lw=1)
    ax.axhline(-1.96, color='red',     lw=0.9, ls='--', alpha=0.55, label='α=5% (C0 mejor)')
    ax.axhline( 1.96, color='green',   lw=0.9, ls='--', alpha=0.55, label='α=5% (C1 mejor)')
    ax.set_xticks(xp)
    ax.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS], fontsize=9)
    ax.set_ylabel('Estadístico DM', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(-7, 7)
    ax.annotate('DM > 0 → C1 mejor\nDM < 0 → C0 mejor',
                xy=(0.02, 0.97), xycoords='axes fraction', fontsize=7,
                va='top', color='gray', bbox=dict(boxstyle='round', fc='white', alpha=0.7))

fig.suptitle('Test Diebold-Mariano por subperiodo — TimesFM C0 vs C1 (HICP Eurozona)\n'
             '** p<0.05  * p<0.10', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig4b_dm_subperiod_europe.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 · Figura Principal — Para la Memoria del TFG

In [ ]:
matplotlib.rcParams.update({'font.family': 'DejaVu Sans',
                             'axes.spines.top': False, 'axes.spines.right': False})

fig = plt.figure(figsize=(17, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.44, wspace=0.34)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

# ── A: Ranking MAE h=12 (top 12 sin naive) ───────────────────────
df_h12_top = (df[(df['horizon']==12) & (df['model']!='naive_europe')]
              .dropna(subset=['MAE']).sort_values('MAE').head(12))
mods_a  = df_h12_top['model'].tolist()
cols_a  = [FAMILY_COLORS.get(m, '#999') for m in mods_a]
ax_a.barh(range(len(mods_a)), df_h12_top['MAE'].values,
          color=cols_a, edgecolor='white', lw=0.5)
ax_a.barh(0, df_h12_top['MAE'].iloc[0], color=cols_a[0], edgecolor='gold', lw=2.5)
ax_a.set_yticks(range(len(mods_a)))
ax_a.set_yticklabels([m.replace('_europe','') for m in mods_a], fontsize=8.5)
ax_a.set_xlabel('MAE (puntos HICP)', fontsize=9)
ax_a.set_title('(A) Ranking MAE — h=12', fontsize=10, fontweight='bold')
ax_a.grid(axis='x', alpha=0.22)
ax_a.invert_yaxis()

# ── B: Perfiles MAE modelos clave (con AutoARIMA) ─────────────────
PROFILE = [
    ('nbeats_europe',          '#1a6ab0', 'Deep: N-BEATS',               'o-',  2.0),
    ('sarima_europe',          '#606060', 'SARIMA (referencia)',          's:',  1.6),
    ('auto_arima_europe',      '#8B4513', 'AutoARIMA (dinámico) ★',       'x--', 1.8),
    ('timesfm_C0_europe',      '#2ca02c', 'TimesFM C0',                   'D-',  2.2),
    ('timesfm_C1_inst_europe', '#9467bd', 'TimesFM C1_inst ★',            'v--', 2.2),
    ('timesfm_C1_full_europe', '#17becf', 'TimesFM C1_full ★★ (best)',    'P-',  2.4),
    ('chronos2_C0_europe',     '#52a852', 'Chronos-2 C0',                 '^-',  1.8),
]
xb = np.arange(len(HORIZONS))
for mn, col, lbl, sty, lw in PROFILE:
    if mn not in metrics: continue
    vals = [metrics[mn].get(f'h{h}', {}).get('MAE') for h in HORIZONS]
    ax_b.plot(xb, vals, sty, color=col, lw=lw, ms=7, label=lbl, zorder=5)
ax_b.set_xticks(xb)
ax_b.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_b.set_ylabel('MAE (puntos HICP)', fontsize=9)
ax_b.set_title('(B) Perfil MAE — Modelos clave', fontsize=10, fontweight='bold')
ax_b.legend(fontsize=7.5, loc='upper left', framealpha=0.75)
ax_b.grid(alpha=0.22)

# ── C: Heatmap delta compacto ─────────────────────────────────────
HM_C = [
    ('Chronos-2 C1_inst', 'chronos2_C0_europe', 'chronos2_C1_inst_europe'),
    ('Chronos-2 C1_mcp',  'chronos2_C0_europe', 'chronos2_C1_mcp_europe'),
    ('Chronos-2 C1_full', 'chronos2_C0_europe', 'chronos2_C1_full_europe'),
    ('TimesFM  C1_inst ★','timesfm_C0_europe',  'timesfm_C1_inst_europe'),
    ('TimesFM  C1_mcp',   'timesfm_C0_europe',  'timesfm_C1_mcp_europe'),
    ('TimesFM  C1_full ★','timesfm_C0_europe',  'timesfm_C1_full_europe'),
    ('TimeGPT  C1_inst',  'timegpt_C0_europe',  'timegpt_C1_inst_europe'),
    ('TimeGPT  C1_mcp',   'timegpt_C0_europe',  'timegpt_C1_mcp_europe'),
    ('TimeGPT  C1_full',  'timegpt_C0_europe',  'timegpt_C1_full_europe'),
]
hmc = np.full((len(HM_C), 4), np.nan)
for i, (_, c0n, c1n) in enumerate(HM_C):
    for j, h in enumerate(HORIZONS):
        m0 = metrics.get(c0n, {}).get(f'h{h}', {}).get('MAE')
        m1 = metrics.get(c1n, {}).get(f'h{h}', {}).get('MAE')
        if m0 and m1: hmc[i, j] = (m1 - m0) / m0 * 100

norm_c = TwoSlopeNorm(vmin=-15, vcenter=0, vmax=100)
imc = ax_c.imshow(hmc, cmap='RdYlGn_r', norm=norm_c, aspect='auto')
ax_c.set_xticks(range(4))
ax_c.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_c.set_yticks(range(len(HM_C)))
ax_c.set_yticklabels([r[0] for r in HM_C], fontsize=8.5)
ax_c.set_title('(C) Δ MAE: (C1−C0)/C0×100%\nVerde=mejora · Rojo=empeora',
               fontsize=10, fontweight='bold')
for i in range(len(HM_C)):
    for j in range(4):
        v = hmc[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 35 else 'black'
            ax_c.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                      fontsize=8.5, color=ct, fontweight='bold')
for b in [3, 6]:
    ax_c.axhline(b - 0.5, color='white', lw=2.5)
plt.colorbar(imc, ax=ax_c, shrink=0.78, label='Δ MAE (%)')

# ── D: DM subperiodo TimesFM C0 vs C1_full ────────────────────────
sub_d = dm_df[(dm_df['model1']=='timesfm_C0_europe') & (dm_df['model2']=='timesfm_C1_full_europe')]
xd = np.arange(len(PERIODS))
wd = 0.18
for i, h in enumerate(HORIZONS):
    dvals, pvals = [], []
    for p in PERIODS:
        row = sub_d[(sub_d['period']==p) & (sub_d['horizon']==h)]
        if len(row) and pd.notna(row['dm_stat'].values[0]):
            dvals.append(float(row['dm_stat'].values[0]))
            pvals.append(float(row['p_value'].values[0]))
        else:
            dvals.append(0.0); pvals.append(1.0)
    ax_d.bar(xd+(i-1.5)*wd, dvals, wd, label=f'h={h}',
             alpha=0.85, color=plt.cm.viridis(i/4), zorder=4)
    for xi, (dv, pv) in enumerate(zip(dvals, pvals)):
        if pv < 0.05:
            ax_d.text(xi+(i-1.5)*wd, dv+(0.22 if dv>=0 else -0.55),
                      '**', ha='center', fontsize=9, fontweight='bold')
        elif pv < 0.10:
            ax_d.text(xi+(i-1.5)*wd, dv+(0.22 if dv>=0 else -0.45),
                      '*', ha='center', fontsize=9)

ax_d.axhline(0,    color='black',   lw=1,   zorder=5)
ax_d.axhline(-1.96,color='#cc0000', lw=0.9, ls='--', alpha=0.6)
ax_d.axhline( 1.96,color='#006600', lw=0.9, ls='--', alpha=0.6)
ax_d.set_xticks(xd)
ax_d.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS], fontsize=8.5)
ax_d.set_ylabel('Estadístico DM', fontsize=9)
ax_d.set_title('(D) DM: TimesFM C0 vs C1_full\nDM>0 → C1_full mejor  |  ** p<0.05',
               fontsize=10, fontweight='bold')
ax_d.legend(fontsize=8, ncol=2)
ax_d.grid(axis='y', alpha=0.22, zorder=0)
ax_d.set_ylim(-6.5, 6.5)

# ── Global ─────────────────────────────────────────────────────────
fig.suptitle(
    'Evaluación Comparativa — HICP Eurozona · Rolling-origin 2021-2024\n'
    'Señales MCP: BCE/GDELT · Señales inst: DFR, ESI, Brent, TTF, EPU · AutoARIMA: selección dinámica · Métrica: MAE',
    fontsize=12, fontweight='bold', y=1.01)

legend_patches = [
    mpatches.Patch(color='#606060', label='Estadístico (SARIMA/SARIMAX)'),
    mpatches.Patch(color='#8B4513', label='AutoARIMA ★ (dinámico: mejor h=1−6, empate h=12 con SARIMA)'),
    mpatches.Patch(color='#1a6ab0', label='Deep Learning (N-BEATS/N-HiTS/LSTM)'),
    mpatches.Patch(color='#2ca02c', label='Foundation C0 (univariante)'),
    mpatches.Patch(color='#9467bd', label='Foundation C1_inst ★ (DFR+ESI+Brent…)'),
    mpatches.Patch(color='#ff7f0e', label='Foundation C1_mcp (BCE+GDELT)'),
    mpatches.Patch(color='#17becf', label='Foundation C1_full ★★ (inst+mcp) — MEJOR modelo'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=7.5,
           frameon=True, bbox_to_anchor=(0.5, -0.055),
           title='Familias de modelos', title_fontsize=9)

plt.savefig(RESULTS / 'fig_MAIN_europe.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('\n=== FIGURA PRINCIPAL GUARDADA: 08_results/fig_MAIN_europe.png ===')

---
## 6 · Tablas de referencia completas

In [ ]:
pivot_mae  = df.pivot(index='model', columns='horizon', values='MAE').reindex(MODEL_ORDER)
pivot_mase = df.pivot(index='model', columns='horizon', values='MASE').reindex(MODEL_ORDER)
pivot_mae.columns  = [f'h={h}' for h in pivot_mae.columns]
pivot_mase.columns = [f'h={h}' for h in pivot_mase.columns]
pivot_mae.index  = [m.replace('_europe','') for m in pivot_mae.index]
pivot_mase.index = [m.replace('_europe','') for m in pivot_mase.index]

print('=== MAE (puntos HICP Eurozona, base 2015=100) ===')
print(pivot_mae.round(4).to_string())
print()
print('=== MASE (escala: naive lag-12 sobre 2002-2020 | <1 = mejor que naive) ===')
print(pivot_mase.round(4).to_string())

In [ ]:
print('=' * 72)
print('SÍNTESIS DE RESULTADOS — HICP EUROZONA')
print('=' * 72)

# Mejor modelo por horizonte
print('\nMEJOR MODELO POR HORIZONTE (MAE):')
for h in HORIZONS:
    row = df[(df['horizon']==h) & (df['model']!='naive_europe')].sort_values('MAE').iloc[0]
    print(f'  h={h:2d}: {row["model"].replace("_europe",""):35s} MAE={row["MAE"]:.4f}')

print('\nCOMPARACIÓN CON SARIMA (referencia baseline):')
sarima_mae = {h: metrics['sarima_europe'][f'h{h}']['MAE'] for h in HORIZONS}
for name in ['auto_arima_europe','timesfm_C0_europe','timesfm_C1_inst_europe','timesfm_C1_full_europe']:
    if name not in metrics: continue
    short = name.replace('_europe','')
    deltas = [f'h={h}: {(metrics[name][f"h{h}"]["MAE"]-sarima_mae[h])/sarima_mae[h]*100:+.1f}%'
              for h in HORIZONS]
    print(f'  {short:35s} ' + '  '.join(deltas))

print('\nAutoARIMA (selección dinámica de órdenes):')
aa = metrics.get('auto_arima_europe', {})
for h in HORIZONS:
    mae_aa = aa.get(f'h{h}', {}).get('MAE')
    mae_sa = sarima_mae[h]
    if mae_aa:
        diff = (mae_aa - mae_sa) / mae_sa * 100
        mark = ' ← mejor' if diff < 0 else ' ← peor'
        print(f'  h={h}: AutoARIMA={mae_aa:.4f}  SARIMA={mae_sa:.4f}  Δ={diff:+.1f}%{mark}')

print('\nSEÑALES C1 — HALLAZGO CENTRAL:')
print('  ★  TimesFM C1_inst: mejora a h=3,6,12 vs C0 (−2% a −4%)')
print('     Globalmente empate estadístico (DM p>0.05). Mejor en pre-shock (h=3**,h=6**,h=12**)')
print('  ★★ TimesFM C1_full: MEJOR modelo del estudio a h=12 (MAE=1.995, primer modelo <2.0)')
print('     Bate a SARIMA en h=6** y h=12**. Empate global vs C0.')
print('  ✗  TimeGPT C1: carry-forward de señales extremas distorsiona predicciones en shock 2022')
print('  ✗  Chronos-2 C1: degradación moderada (+1%−5%) en todas las condiciones')

print('\nAutoARIMA vs SARIMA FIJO — HALLAZGO METODOLÓGICO:')
print('  ★ AutoARIMA mejora SARIMA en h=1,3,6 (entre −8.9% y −6.3%)')
print('  ✗ AutoARIMA algo peor a h=12 (SARIMA=2.411 vs AutoARIMA=2.510, Δ=+4.1%)')
print('  → La selección dinámica de órdenes no garantiza mejora en horizontes largos')
print('    (posible overfitting al ajustar parámetros en cada ventana rolling)')

print('\nTODOS LOS MODELOS BATEN AL NAIVE (DM p<0.001 en todos los horizontes)')
print('=' * 72)